# Chapter 5 AFC-BJK target construction

**Status:** canonical revised-method experiment  
**Thesis sections:** 5.1.1–5.1.2

Chapter 5 changes the reference solution, rather than merely tuning a model. This notebook retains the SPDE 3 definition and makes the AFC-BJK/SUPG relationship visible. The limiter implementation and file persistence are packaged because they are lengthy algorithms, not the scientific argument.

## 1. SPDE 3: geometry, flow direction, and discontinuous boundary data

The inflow/outflow geometry creates the sharp layer used in the revised study. Keep this cell next to results: it is the concise, executable definition of the problem studied in the thesis.

In [ ]:
import numpy as np
import ufl
from dolfinx import default_scalar_type, fem, mesh as dmesh
from mpi4py import MPI

mesh = dmesh.create_unit_square(MPI.COMM_WORLD, 64, 64)
V = fem.functionspace(mesh, ("CG", 1))
u_h = fem.Function(V, name="supg_solution")
x = ufl.SpatialCoordinate(mesh)
eps = fem.Constant(mesh, default_scalar_type(1e-8))
b = ufl.as_vector((fem.Constant(mesh, np.cos(-np.pi / 3)), fem.Constant(mesh, np.sin(-np.pi / 3))))

g = ufl.conditional(
    ufl.ge(x[1], 0.7 + ufl.sin(-ufl.pi / 3) / ufl.cos(-ufl.pi / 3) * x[0]), 1, 0
) * ufl.conditional(ufl.Or(ufl.eq(x[1], 0), ufl.eq(x[0], 1)), 0, 1)
mesh.topology.create_connectivity(1, 2)
boundary_dofs = fem.locate_dofs_topological(V, 1, dmesh.exterior_facet_indices(mesh.topology))
g_h = fem.Function(V)
g_h.interpolate(fem.Expression(g, V.element.interpolation_points()))
bc = fem.dirichletbc(g_h, boundary_dofs)


## 2. Use AFC-BJK as the reference solution

AFC-BJK is computed once for this mesh and becomes the target $u_{\mathrm{BJK}}$. The learning objective is then $\int_\Omega (u_\tau-u_{\mathrm{BJK}})^2\,dx$. The calls below state the algorithmic choice while keeping the limiter’s flux-correction loops in the tested `supgml.stabilization` implementation.

In [ ]:
from supgml.supg import AdjointSUPGSolver, ConvectionDiffusionProblem

# The AFC solver supplies u_bjk for this mesh, coefficient field, and boundary data.
# u_bjk = BJKAFC(...).solve()
#
# Once u_bjk is available, the SUPG optimization problem is explicit:
problem = ConvectionDiffusionProblem(mesh, V, u_h, eps, b, None, fem.Constant(mesh, 0.0), None, [bc])
u_bjk = fem.Function(V, name="afc_bjk_target")  # load/compute before optimization
objective = (u_h - u_bjk) ** 2 * ufl.dx
adjoint_solver = AdjointSUPGSolver(problem, objective)


## 3. Persist the scientific artefacts

Save the mesh, $u_{\mathrm{BJK}}$, optimized cellwise parameters, interior mask, and named feature matrix as one graph case. `CaseRepository` provides the on-disk convention; notebook 07 consumes that case. Perturbation analysis belongs in notebook 08.